<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Exercises_XP_RAG_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: RAG with LangChain (Student)

## 0) Setup


In [ ]:
!pip install -U "pyarrow<17.0.0" "datasets==2.19.0" langchain-community langchain-huggingface langchain-core langchain-text-splitters transformers faiss-cpu

In [ ]:
from typing import List

from datasets import load_dataset
from transformers import pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA


## 1) Load dataset and convert to Documents


In [ ]:
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]"
text_column = "text"
source_column = "source"

ds = load_dataset(dataset_name, split=split)

documents: List[Document] = []
for i, row in enumerate(ds):
    documents.append(
        Document(
            page_content=row[text_column],
            metadata={"source": row[source_column]}
        )
    )

print("Documents:", len(documents))
print("Example:", documents[0].metadata)
print(documents[0].page_content[:350])

## 2) Split into chunks


In [ ]:
chunk_size = 512
chunk_overlap = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

splits = splitter.split_documents(documents)
print("Chunks:", len(splits))
print("First chunk metadata:", splits[0].metadata)
print("First chunk content:", splits[0].page_content[:350])

## 3) Vector store + retriever (FAISS)


In [ ]:
from langchain_community.vectorstores import FAISS, DistanceStrategy

embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings,
    distance_strategy=DistanceStrategy.COSINE
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever ready")

## 4) Build the RAG chain


In [ ]:
import torch
from transformers import (AutoTokenizer, pipeline, AutoModelForSeq2SeqLM)
from langchain_community.llms import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA

llm_id = "google/flan-t5-small"

# Load model and tokenizer explicitly to avoid pipeline task registry issues
tokenizer = AutoTokenizer.from_pretrained(llm_id)
model = AutoModelForSeq2SeqLM.from_pretrained(llm_id)

hf_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device=-1,
    max_length=512,
    do_sample=False
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

print("RAG chain ready")

## 5) Demo: RAG vs no-RAG


In [ ]:
q = "How can I retrieve a model from the Hugging Face Hub?"

# No-RAG (LLM only)
no_rag_prompt = (
    "Answer the question. If you are not sure, say you are not sure.\n\n"
    f"Question: {q}\n"
    "Answer:"
)
no_rag_answer = hf_pipeline(no_rag_prompt)[0]["generated_text"]

# RAG
# On passe la question dans un dictionnaire comme attendu par RetrievalQA
rag_result = qa({"query": q})

print("Q:", q)
print("\nNo-RAG answer:\n", no_rag_answer)
print("\nRAG answer:\n", rag_result["result"])
print("\nSources:")
for d in rag_result["source_documents"]:
    print("-", d.metadata.get("source"))

In [ ]:
question = "How can I retrieve a model from the Hugging Face Hub?"

docs = retriever.invoke(question)

print(f"Nombre de documents trouvés : {len(docs)}\n")

for i, doc in enumerate(docs, 1):
    print(f"Document {i}")
    print("-" * 50)
    print(doc.page_content[:500])
    print()